In [ ]:
import pandas as pd
import re
import nltk
import json
from nltk.corpus import stopwords
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score
from datetime import datetime, timedelta
from difflib import SequenceMatcher
import time
import requests
from datetime import datetime, timedelta
from collections import Counter
nltk.download("stopwords")

In [ ]:
NEWSAPI_KEY = "ed945ddfa03f4961b9bf24d60d3eeb45"
NEWSAPI_BASE = "https://newsapi.org/v2/everything"

SOURCE_DATABASE = {
    # === Highly Reliable (International) ===
    "reuters.com":          {"score": 0.97, "bias": "center",       "category": "mainstream",  "notes": "Global newswire, high factual reporting"},
    "apnews.com":           {"score": 0.96, "bias": "center",       "category": "mainstream",  "notes": "Associated Press, non-profit news cooperative"},
    "bbc.com":              {"score": 0.92, "bias": "center-left",  "category": "mainstream",  "notes": "UK public broadcaster"},
    "bbc.co.uk":            {"score": 0.92, "bias": "center-left",  "category": "mainstream",  "notes": "UK public broadcaster"},
    "theguardian.com":      {"score": 0.88, "bias": "center-left",  "category": "mainstream",  "notes": "British daily newspaper"},
    "nytimes.com":          {"score": 0.87, "bias": "center-left",  "category": "mainstream",  "notes": "Major US newspaper"},
    "washingtonpost.com":   {"score": 0.86, "bias": "center-left",  "category": "mainstream",  "notes": "Major US newspaper"},
    "economist.com":        {"score": 0.92, "bias": "center-right", "category": "mainstream",  "notes": "UK weekly news magazine"},
    "ft.com":               {"score": 0.91, "bias": "center",       "category": "mainstream",  "notes": "Financial Times"},
    "wsj.com":              {"score": 0.88, "bias": "center-right", "category": "mainstream",  "notes": "Wall Street Journal"},
    "bloomberg.com":        {"score": 0.90, "bias": "center",       "category": "mainstream",  "notes": "Business and financial news"},
    "npr.org":              {"score": 0.91, "bias": "center-left",  "category": "mainstream",  "notes": "US public radio"},
    "pbs.org":              {"score": 0.90, "bias": "center-left",  "category": "mainstream",  "notes": "US public broadcaster"},
    "politifact.com":       {"score": 0.93, "bias": "center",       "category": "fact-check",  "notes": "Pulitzer-winning fact checker"},
    "snopes.com":           {"score": 0.91, "bias": "center",       "category": "fact-check",  "notes": "Oldest fact-checking site"},
    "factcheck.org":        {"score": 0.92, "bias": "center",       "category": "fact-check",  "notes": "Non-partisan fact checker"},

    # === Indian News Sources ===
    "thehindu.com":         {"score": 0.88, "bias": "center-left",  "category": "mainstream",  "notes": "Major Indian English daily"},
    "hindustantimes.com":   {"score": 0.82, "bias": "center",       "category": "mainstream",  "notes": "Major Indian English daily"},
    "indianexpress.com":    {"score": 0.85, "bias": "center-left",  "category": "mainstream",  "notes": "Indian English newspaper"},
    "ndtv.com":             {"score": 0.80, "bias": "center-left",  "category": "mainstream",  "notes": "Indian news channel"},
    "timesofindia.com":     {"score": 0.78, "bias": "center",       "category": "mainstream",  "notes": "India's largest English newspaper"},
    "scroll.in":            {"score": 0.82, "bias": "center-left",  "category": "mainstream",  "notes": "Indian digital news publication"},
    "thewire.in":           {"score": 0.78, "bias": "left",         "category": "mainstream",  "notes": "Indian independent news"},
    "alt-news.in":          {"score": 0.88, "bias": "center-left",  "category": "fact-check",  "notes": "Indian fact-checking website"},
    "boomlive.in":          {"score": 0.87, "bias": "center",       "category": "fact-check",  "notes": "Indian fact-checking website"},
    "livemint.com":         {"score": 0.82, "bias": "center",       "category": "mainstream",  "notes": "Indian financial newspaper"},
    "businessstandard.com": {"score": 0.83, "bias": "center",       "category": "mainstream",  "notes": "Indian business newspaper"},
    "zeenews.india.com":    {"score": 0.65, "bias": "right",        "category": "mainstream",  "notes": "Indian news with right-leaning bias"},
    "republic world.com":   {"score": 0.55, "bias": "right",        "category": "tabloid",     "notes": "Sensationalist Indian news channel"},
    "opindia.com":          {"score": 0.35, "bias": "right",        "category": "partisan",    "notes": "Strong right-wing bias, opinion-heavy"},
    "postcard.news":        {"score": 0.15, "bias": "right",        "category": "conspiracy",  "notes": "Frequently publishes misinformation"},

    # === Mixed / Moderate Reliability ===
    "cnn.com":              {"score": 0.78, "bias": "center-left",  "category": "mainstream",  "notes": "Major US cable news"},
    "foxnews.com":          {"score": 0.60, "bias": "right",        "category": "mainstream",  "notes": "Right-leaning, mixed factual record"},
    "msnbc.com":            {"score": 0.68, "bias": "left",         "category": "mainstream",  "notes": "Left-leaning US cable news"},
    "dailymail.co.uk":      {"score": 0.45, "bias": "right",        "category": "tabloid",     "notes": "British tabloid, low factual rating"},
    "nypost.com":           {"score": 0.50, "bias": "right",        "category": "tabloid",     "notes": "US tabloid-style newspaper"},
    "huffpost.com":         {"score": 0.68, "bias": "left",         "category": "mainstream",  "notes": "Left-leaning US digital news"},
    "vice.com":             {"score": 0.70, "bias": "left",         "category": "mainstream",  "notes": "Youth-oriented news"},
    "buzzfeednews.com":     {"score": 0.72, "bias": "center-left",  "category": "mainstream",  "notes": "Digital news outlet"},

    # === Satire (Not Fake, but Often Misread) ===
    "theonion.com":         {"score": 0.10, "bias": "center",       "category": "satire",      "notes": "SATIRE — Not real news"},
    "babylonbee.com":       {"score": 0.10, "bias": "right",        "category": "satire",      "notes": "SATIRE — Not real news"},
    "thebeaverton.com":     {"score": 0.10, "bias": "center",       "category": "satire",      "notes": "SATIRE — Canadian, not real news"},

    # === Known Misinformation / Conspiracy Sites ===
    "infowars.com":         {"score": 0.05, "bias": "right",        "category": "conspiracy",  "notes": "Conspiracy theories, repeatedly debunked"},
    "naturalnews.com":      {"score": 0.08, "bias": "right",        "category": "conspiracy",  "notes": "Health misinformation, conspiracy theories"},
    "beforeitsnews.com":    {"score": 0.05, "bias": "right",        "category": "conspiracy",  "notes": "Conspiracy and misinformation hub"},
    "worldnewsdailyreport.com": {"score": 0.02, "bias": "unknown",  "category": "fake",        "notes": "Fabricated news site"},
    "empirenews.net":       {"score": 0.02, "bias": "unknown",      "category": "fake",        "notes": "Known fake news site"},
}

In [ ]:
#english dataset preprocesing
nltk.download("stopwords")
df = pd.read_csv("engnews.csv")
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)
stop_words = set(stopwords.words("english"))
def clean_engtext(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
df["clean_text"] = df["text"].apply(clean_engtext)
df = df[df["clean_text"].str.strip() != ""]
df = df[["clean_text", "FAKE"]]
df.to_csv("clean_english_news_dataset.csv", index=False)
#print("Clean dataset saved")
#print("Total rows:", len(df))

#Hindi dataset preprocessing
df = pd.read_csv("hindinews.csv", encoding="utf-8")
eng_stop = set(stopwords.words("english"))
hinglish_stop = set(stopwords.words("hinglish"))
stop_words = eng_stop.union(hinglish_stop)
def clean_hinditext(text):
    text = str(text)
    text = transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
df["clean_text"] = df["text"].apply(clean_hinditext)
df = df[["clean_text", "FAKE"]]
df.to_csv("clean_hindi_news_dataset.csv", index=False, encoding="utf-8-sig")
#print("Clean dataset created successfully")

In [ ]:
df1 = pd.read_csv("clean_english_news_dataset.csv")
df2 = pd.read_csv("clean_hindi_news_dataset.csv")
merged = pd.concat([df1, df2], ignore_index=True)
merged.to_csv("merged_ds.csv", index=False)
#print("done")

In [ ]:
df = pd.read_csv("merged_ds.csv")
X = df['clean_text']  
y = df['FAKE']  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


amodel = MultinomialNB()
amodel.fit(X_train_tfidf, y_train)

y_pred = amodel.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
def detect_lang(text):
    # Hindi characters range
    if re.search(r'[\u0900-\u097F]', text):
        return "hindi"
    return "eng"

def fakenews_score(news_text):
    if news_text is None:
        return {"Probability" : 0 , "Label" : "None"} 
    else:
        language = detect_lang(news_text)
        if language=="hindi":
            clean= clean_hinditext(news_text)
        elif language=="eng":
            clean = clean_engtext(news_text)
        #else:
          #  print("Error in determining language of the news")

    
        vec = vectorizer.transform([clean])
        fake_prob = amodel.predict_proba(vec)[0][1]
        ml_score = 1 - fake_prob

        '''if ml_score > 0.7:
            label = "LIKELY TRUE"
        elif ml_score > 0.4:
            label = "UNCERTAIN"
        else:
            label = "LIKELY FAKE"'''
    
        return ml_score


In [ ]:
def fakenews_score(news_text):
    vec = vectorizer.transform([clean])
    fake_prob = amodel.predict_proba(vec)[0][1]
    ml_score = 1 - fake_prob
    return ml_score

In [ ]:
def source_score(source_name: str):
    source_name = source_name.lower().strip()

    if source_name in SOURCE_DATABASE:
        return SOURCE_DATABASE[source_name]["score"]
    else:
        return 0.1 

In [ ]:
def extract_keywords(headline: str, max_keywords: int = 5) -> str:
    stopw = set(stopwords.words("english"))
    words = re.findall(r'\b[a-zA-Z]{3,}\b', headline.lower())
    keywords = [w for w in words if w not in stopw]
    unique = list(dict.fromkeys(keywords))[:max_keywords]  # FIXED
    return " ".join(unique)

def search_newsapi(query: str, days_back: int = 7) -> list:
    from_date = (datetime.utcnow() - timedelta(days=days_back)).strftime("%Y-%m-%d")
    params = {
        "q": query, "from": from_date, "sortBy": "relevancy",
        "language": "en", "pageSize": 30, "apiKey": NEWSAPI_KEY,}
    try:
        r = requests.get(NEWSAPI_BASE, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        return [
            {"title": a.get("title",""), "source": a.get("source",{}).get("name","Unknown"),
             "url": a.get("url",""), "published": a.get("publishedAt","")}
            for a in data.get("articles", [])
        ]
    except requests.RequestException:
        return []

def multiplatform_score(headline: str, days_back: int = 7) -> dict:
    keywords = extract_keywords(headline)
    articles = search_newsapi(keywords, days_back)
    unique_sources = list({a["source"] for a in articles})
    times=len(unique_sources)
    score = min(times/10, 1.0) 
    return score

In [ ]:
f1 = pd.read_csv("data1.csv")
df2 = pd.read_csv("data2.csv")

df2["label"] = df2["label"].replace({"clickbait": 1,"news": 0})
df2 = df2.rename(columns={"title": "headline","label": "clickbait"})
df2 = df2[["headline", "clickbait"]]
df2.to_csv("data2_updated.csv", index=False)

df2 = pd.read_csv("data2_updated.csv")
merged = pd.concat([df1, df2], ignore_index=True)
merged.to_csv("merged_data.csv", index=False)

df = pd.read_csv("merged_data.csv")
df.dropna(inplace=True)
keep_words = {"what","why","how","when","who","which","you","your","we","our","now","today","must","need","watch","see","read","know","shocking","secret","revealed","amazing","viral",
    "top","best","ways","reasons","things","tricks","tips","warning","truth","hidden","risk"}
stop_words = set(stopwords.words("english"))
stop_words = stop_words - keep_words

def extra_features(text):
    text = str(text)
    exclam = text.count("!")
    ques = text.count("?")
    caps_words = sum(1 for ch in text if ch.isupper())
    has_number = 1 if re.search(r'\d', text) else 0
    return exclam, ques, caps_words, has_number


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df[["exclam","ques","caps","number"]] = df["headline"].apply(
    lambda x: pd.Series(extra_features(x)))

df["headline"] = df["headline"].apply(clean_text)
df = df[df["headline"] != ""]
df.to_csv("clean_data.csv", index=False)

df = pd.read_csv("clean_data.csv")
X = df["headline"]
y = df["clickbait"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


bmodel = MultinomialNB()
bmodel.fit(X_train_tfidf, y_train)

y_pred = bmodel.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

In [ ]:
df = pd.read_csv("clean_data.csv")
clickbait_df = df[df["clickbait"] == 1]
normal_df    = df[df["clickbait"] == 0]
clickbait_words = Counter()
normal_words = Counter()
for text in clickbait_df["headline"]:
    words = str(text).split()
    clickbait_words.update(words)
for text in normal_df["headline"]:
    words = str(text).split()
    normal_words.update(words)

data = []
all_words = set(clickbait_words.keys()) | set(normal_words.keys())
for word in all_words:
    cb_freq = clickbait_words[word]
    nb_freq = normal_words[word]
    if cb_freq + nb_freq >= 5:
        score = cb_freq - nb_freq
        data.append([word, cb_freq, nb_freq, score])

result = pd.DataFrame(data,columns=["word","clickbait_freq","nonclickbait_freq","score"])
result = result.sort_values(by="score",ascending=False)
top500 = result.head(500)
top500.to_csv("sus_words.csv", index=False)

manual = ["shocking", "secret", "revealed", "viral", "unbelievable", "amazing", "must", "urgent", "warning", "truth", "exposed", "banned", "hidden", "insane", "crazy", "surprising", "incredible", "mindblowing", "epic", "stunning", "danger", "risk", "hate", "destroy", "scam", "fraud", "fake", "alert", "watch", "see", "look", "now", "today", "finally", "breaking", "top", "best", "ways", "reasons", "things", "tricks", "tips", "hack", "simple", "easy", "fast", "instant", "you", "your", "why", "how", "what", "won't", "believe", "next", "number", "experts", "doctor", "guaranteed"]

def sus_score(headline):
    text = headline.lower()
    l=len(text)
    count = 0
    import pandas as pd
    df = pd.read_csv("sus_words.csv")
    sus_words = df["word"].str.lower().tolist()
    sus_words.append(manual)
    
    for word in text:
        if word in sus_words:
            count += 1
    score = count/l
    return score

def ex_score(headline):
    c = headline.count("!")
    score = min(c* 0.3, 1)
    return c

def cap_score(headline):
    letters = [ch for ch in headline if ch.isalpha()]
    upper = sum(1 for ch in letters if ch.isupper())
    if len(letters) > 0:
        capital_score = (upper / len(letters)) 
    else:
        capital_score = 0
    return capital_score

def ml_score(headline):
    text=clean_text(headline)
    vec = vectorizer.transform([text])
    fake_prob = bmodel.predict_proba(vec)[0][1]
    ml_score = fake_prob
    return ml_score

def clickbait_score(headline):
    c=cap_score(headline)
    e=ex_score(headline)
    m=ml_score(headline)
    s=sus_score(headline)
    total=0.05*(c+e)+0.1*s+0.8*m
    return total


In [ ]:
def total_score(f,c,m,s):
    total=(0.4*f+0.4*(1-c)+0.1*m+0.1*s)*100
    return total

In [ ]:
def get_news(api_key):
    url = "https://newsapi.org/v2/top-headlines"
    params = {
        "country": "us",
        "apiKey": api_key}
    response = requests.get(url, params=params)
    return response.json()
news_list = get_news(NEWSAPI_KEY)

print(news_list)